In [ ]:
#Capstone Project 3
#Smart Marketing Prediction System (ML Pipeline Project)
#Scenario
#A fast-growing e-commerce company called ShopEasy is struggling with inefficient marketing campaigns.
#Every day thousands of users visit their website. The marketing team spends a large amount of money showing ads, discounts, and promotional emails, but they don't know which customers are actually likely to buy something.

#Currently:
#Many customers browse but never purchase
#Marketing money is wasted on the wrong users
#The company wants to predict purchase probability
#The data science team has been asked to build a machine learning system that predicts whether a customer will purchase a product during a session.
#If the system predicts high probability of purchase, the system will:
#show personalized product recommendations
#offer targeted discounts
#prioritize marketing campaigns
#If the system predicts low probability, the company will avoid spending marketing resources.
#However, the dataset contains both numerical and categorical features, so the data science team must design a complete ML pipeline.

#Business Objective
#Build a machine learning model that predicts whether a user will purchase (1) or not purchase (0) during a website session.
#The model must be implemented using scikit-learn pipelines, including:
#Encoding techniques
#Feature preprocessing
#Model training
#Model selection
#Hyperparameter tuning

#step 1: Import libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import classification_report, accuracy_score
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

In [ ]:
#step 2: Load data
df = pd.read_csv("DatasetCapstoneProject3.csv")

In [ ]:
df.head()

,CustomerID,Age,Gender,Device,Traffic_Source,Time_on_Website,Pages_Visited,Ad_Clicks,Previous_Purchases,Purchased
0,1,23,Male,Mobile,Social Media,5,3,1,0,0
1,2,35,Female,Desktop,Search Engine,12,8,3,2,1
2,3,29,Male,Tablet,Social Media,8,5,2,1,0
3,4,41,Female,Mobile,Email Campaign,15,10,4,3,1
4,5,22,Female,Desktop,Direct,4,2,0,0,0


In [ ]:
df.columns

Index(['CustomerID', 'Age', 'Gender', 'Device', 'Traffic_Source',
       'Time_on_Website', 'Pages_Visited', 'Ad_Clicks', 'Previous_Purchases',
       'Purchased'],
      dtype='object')

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15 entries, 0 to 14
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   CustomerID          15 non-null     int64 
 1   Age                 15 non-null     int64 
 2   Gender              15 non-null     object
 3   Device              15 non-null     object
 4   Traffic_Source      15 non-null     object
 5   Time_on_Website     15 non-null     int64 
 6   Pages_Visited       15 non-null     int64 
 7   Ad_Clicks           15 non-null     int64 
 8   Previous_Purchases  15 non-null     int64 
 9   Purchased           15 non-null     int64 
dtypes: int64(7), object(3)
memory usage: 1.3+ KB


In [ ]:
#step 3: Define Features and Target
X = df.drop(["Purchased","CustomerID"], axis=1)
y = df["Purchased"]


#step 4: Define Numerical and Categorical Features
numerical_features = [
    'Age',
    'Time_on_Website',
    'Pages_Visited',
    'Ad_Clicks',
    'Previous_Purchases'
]

categorical_features = [
    'Gender',
    'Device',
    'Traffic_Source'
]


#step 5: Create Preprocessing Pipelines
numeric_transformer = Pipeline(steps=[
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])


#step 6: Column Transformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numerical_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)


#step 7: Machine Learning Pipeline
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', LogisticRegression(max_iter=1000))
])


#step 8: Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

#step 9: Hyperparameter Grid
param_grid = [{
        'model': [LogisticRegression()],
        'model__C': [0.01, 0.1, 1, 10]
    },

    {
        'model': [DecisionTreeClassifier()],
        'model__max_depth': [3,5,10]
    }]


#step 10: GridSearchCV
grid_search = GridSearchCV(
    pipeline,
    param_grid,
    cv=3,
    scoring='accuracy'
)

grid_search.fit(X_train, y_train)


#step 11: Best Model
print("\nBest Parameters:", grid_search.best_params_)
print("Best Cross Validation Score:", grid_search.best_score_)


#step 12: Model Evaluation
best_model = grid_search.best_estimator_

y_pred = best_model.predict(X_test)

print("\nModel Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))


#step 13: Predict for New Customer
new_customer = pd.DataFrame({
    'Age':[30],
    'Gender':['Male'],
    'Device':['Mobile'],
    'Traffic_Source':['Ads'],
    'Time_on_Website':[8],
    'Pages_Visited':[5],
    'Ad_Clicks':[2],
    'Previous_Purchases':[1]
})

prediction = best_model.predict(new_customer)

print("\nPrediction for New Customer:", prediction)

if prediction[0] == 1:
    print("Customer is likely to PURCHASE")
else:
    print("Customer is NOT likely to purchase")


Best Parameters: {'model': DecisionTreeClassifier(), 'model__max_depth': 3}
Best Cross Validation Score: 1.0

Model Accuracy: 1.0

Classification Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00         2
           1       1.00      1.00      1.00         1

    accuracy                           1.00         3
   macro avg       1.00      1.00      1.00         3
weighted avg       1.00      1.00      1.00         3


Prediction for New Customer: [0]
Customer is NOT likely to purchase
